In [7]:
# ============================================================
# CELL 1: SETUP & IMPORTS
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Imports ολοκληρώθηκαν")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports ολοκληρώθηκαν


In [8]:
# ============================================================
# CELL 2: ΦΟΡΤΩΣΗ ΟΛΩΝ ΤΩΝ 4 DATASETS (με flexible parsing)
# ============================================================

paths = {
    'BESS':        '/content/drive/MyDrive/Final_RUn/239-Site_DKA_Totals-BESS.csv',
    'Site_Demand': '/content/drive/MyDrive/Final_RUn/240-Site_DKA_Totals-Site_Demand.csv',
    'PV':          '/content/drive/MyDrive/Final_RUn/241-Site_DKA_Totals-PV.csv',
    'Grid':        '/content/drive/MyDrive/Final_RUn/242-Site_DKA_Totals-Grid.csv'
}

datasets = {}

for name, path in paths.items():
    print(f"Φόρτωση {name}...")
    try:
        # engine='python' είναι πιο ανεκτικό σε ανομοιόμορφες γραμμές
        # on_bad_lines='skip' παραλείπει προβληματικές γραμμές
        df_temp = pd.read_csv(
            path,
            engine='python',
            on_bad_lines='skip'
        )
        datasets[name] = df_temp
        print(f"  Επιτυχία: {len(df_temp):,} γραμμές, {len(df_temp.columns)} στήλες")
    except Exception as e:
        print(f"  ΣΦΑΛΜΑ: {e}")

print(f"\nΦορτώθηκαν {len(datasets)} datasets")

Φόρτωση BESS...
  Επιτυχία: 250,740 γραμμές, 14 στήλες
Φόρτωση Site_Demand...
  Επιτυχία: 182,349 γραμμές, 13 στήλες
Φόρτωση PV...
  Επιτυχία: 182,295 γραμμές, 13 στήλες
Φόρτωση Grid...
  Επιτυχία: 182,289 γραμμές, 13 στήλες

Φορτώθηκαν 4 datasets


In [9]:
# ============================================================
# CELL 3: ΕΞΕΡΕΥΝΗΣΗ ΣΤΗΛΩΝ ΚΑΘΕ DATASET
# ============================================================

for name, df in datasets.items():
    print("="*70)
    print(f" DATASET: {name}")
    print("="*70)
    print(f"Γραμμές: {len(df):,}")
    print(f"Στήλες: {len(df.columns)}")
    print(f"\nΛίστα στηλών:")
    for i, col in enumerate(df.columns, 1):
        dtype = df[col].dtype
        print(f"  {i:2d}. {col:<50} ({dtype})")
    print()

 DATASET: BESS
Γραμμές: 250,740
Στήλες: 14

Λίστα στηλών:
   1. timestamp                                          (object)
   2. Reactive_Power                                     (float64)
   3. Active_Power                                       (float64)
   4. Apparent_Power                                     (float64)
   5. State_of_Charge                                    (float64)
   6. Wind_Speed                                         (float64)
   7. Weather_Temperature_Celsius                        (float64)
   8. Weather_Relative_Humidity                          (float64)
   9. Global_Horizontal_Radiation                        (float64)
  10. Diffuse_Horizontal_Radiation                       (float64)
  11. Wind_Direction                                     (float64)
  12. Weather_Daily_Rainfall                             (float64)
  13. Radiation_Global_Tilted                            (float64)
  14. Radiation_Diffuse_Tilted                           (float64)

 DAT

In [10]:
# ============================================================
# CELL 4: ΧΡΟΝΙΚΟ ΕΥΡΟΣ & ΑΝΑΛΥΣΗ ΔΕΙΓΜΑΤΟΛΗΨΙΑΣ
# ============================================================

# Εντοπισμός timestamp column σε κάθε dataset
print("="*70)
print(" ΧΡΟΝΙΚΟ ΕΥΡΟΣ & ΑΝΑΛΥΣΗ")
print("="*70)

time_info = {}

for name, df in datasets.items():
    print(f"\n--- {name} ---")

    # Ψάχνουμε για timestamp column
    time_cols = [col for col in df.columns
                 if any(kw in col.lower() for kw in ['time', 'date', 'stamp'])]

    if not time_cols:
        print("  Δεν βρέθηκε timestamp column!")
        continue

    time_col = time_cols[0]
    print(f"  Timestamp column: '{time_col}'")

    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    valid_times = df[time_col].dropna()

    if len(valid_times) == 0:
        print("  Σφάλμα στη μετατροπή ημερομηνιών")
        continue

    start = valid_times.min()
    end = valid_times.max()
    duration_days = (end - start).days

    # Sampling frequency (διαφορά μεταξύ διαδοχικών timestamps)
    time_diffs = valid_times.sort_values().diff().dropna()
    if len(time_diffs) > 0:
        median_diff = time_diffs.median()
        sampling_minutes = median_diff.total_seconds() / 60
    else:
        sampling_minutes = None

    print(f"  Από: {start}")
    print(f"  Έως: {end}")
    print(f"  Διάρκεια: {duration_days} ημέρες ({duration_days/365:.1f} χρόνια)")
    print(f"  Sampling: ~{sampling_minutes:.1f} λεπτά")

    time_info[name] = {
        'time_col': time_col,
        'start': start,
        'end': end,
        'duration_days': duration_days,
        'sampling_minutes': sampling_minutes
    }

# Συγκριτικός πίνακας
print(f"\n{'='*70}")
print(" ΣΥΓΚΡΙΤΙΚΟΣ ΠΙΝΑΚΑΣ")
print("="*70)
print(f"{'Dataset':<15} {'Start':<20} {'End':<20} {'Sampling':>10}")
print("-"*70)
for name, info in time_info.items():
    print(f"{name:<15} {str(info['start'])[:19]:<20} {str(info['end'])[:19]:<20} "
          f"{info['sampling_minutes']:>7.1f} min")

# Έλεγχος συμβατότητας ημερομηνιών (κοινή χρονική περίοδος)
if len(time_info) > 1:
    common_start = max(info['start'] for info in time_info.values())
    common_end = min(info['end'] for info in time_info.values())
    common_days = (common_end - common_start).days

    print(f"\n{'='*70}")
    print(" ΚΟΙΝΗ ΧΡΟΝΙΚΗ ΠΕΡΙΟΔΟΣ")
    print("="*70)
    print(f"Από: {common_start}")
    print(f"Έως: {common_end}")
    print(f"Διάρκεια: {common_days} ημέρες ({common_days/365:.1f} χρόνια)")

 ΧΡΟΝΙΚΟ ΕΥΡΟΣ & ΑΝΑΛΥΣΗ

--- BESS ---
  Timestamp column: 'timestamp'
  Από: 2023-12-07 13:20:00
  Έως: 2026-04-30 03:45:00
  Διάρκεια: 874 ημέρες (2.4 χρόνια)
  Sampling: ~5.0 λεπτά

--- Site_Demand ---
  Timestamp column: 'timestamp'
  Από: 2023-12-07 13:10:00
  Έως: 2025-09-04 12:15:00
  Διάρκεια: 636 ημέρες (1.7 χρόνια)
  Sampling: ~5.0 λεπτά

--- PV ---
  Timestamp column: 'timestamp'
  Από: 2023-12-07 13:15:00
  Έως: 2025-09-04 12:15:00
  Διάρκεια: 636 ημέρες (1.7 χρόνια)
  Sampling: ~5.0 λεπτά

--- Grid ---
  Timestamp column: 'timestamp'
  Από: 2023-12-07 13:20:00
  Έως: 2025-09-04 12:10:00
  Διάρκεια: 636 ημέρες (1.7 χρόνια)
  Sampling: ~5.0 λεπτά

 ΣΥΓΚΡΙΤΙΚΟΣ ΠΙΝΑΚΑΣ
Dataset         Start                End                    Sampling
----------------------------------------------------------------------
BESS            2023-12-07 13:20:00  2026-04-30 03:45:00      5.0 min
Site_Demand     2023-12-07 13:10:00  2025-09-04 12:15:00      5.0 min
PV              2023-12-07 13:15

In [11]:
# ============================================================
# CELL 5: ΠΡΩΤΕΣ ΓΡΑΜΜΕΣ ΚΑΘΕ DATASET
# ============================================================

for name, df in datasets.items():
    print("="*70)
    print(f" {name} - ΠΡΩΤΕΣ 3 ΓΡΑΜΜΕΣ")
    print("="*70)
    print(df.head(3).to_string())
    print()

 BESS - ΠΡΩΤΕΣ 3 ΓΡΑΜΜΕΣ
            timestamp  Reactive_Power  Active_Power  Apparent_Power  State_of_Charge  Wind_Speed  Weather_Temperature_Celsius  Weather_Relative_Humidity  Global_Horizontal_Radiation  Diffuse_Horizontal_Radiation  Wind_Direction  Weather_Daily_Rainfall  Radiation_Global_Tilted  Radiation_Diffuse_Tilted
0 2023-12-07 13:20:00             NaN      2.302290             NaN              NaN         NaN                    40.718098                  16.468817                  1127.250122                    192.246567       50.439892                     0.0              1138.044312                214.660950
1 2023-12-07 13:25:00             NaN      3.791621             NaN              NaN         NaN                    40.328514                  16.284969                  1199.614136                    275.590546       50.809429                     0.0              1131.120361                252.383423
2 2023-12-07 13:30:00             NaN      0.453575             Na

In [13]:
# ============================================================
# CELL 6: ΣΤΑΤΙΣΤΙΚΑ ΑΡΙΘΜΗΤΙΚΩΝ ΣΤΗΛΩΝ
# ============================================================

for name, df in datasets.items():
    print("="*70)
    print(f" {name} - ΣΤΑΤΙΣΤΙΚΑ")
    print("="*70)

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if not numeric_cols:
        print("  Δεν υπάρχουν αριθμητικές στήλες")
        continue

    stats = pd.DataFrame({
        'Min': df[numeric_cols].min().round(3),
        'Max': df[numeric_cols].max().round(2),
        'Mean': df[numeric_cols].mean().round(2),
        'Std': df[numeric_cols].std().round(2),
        'Missing %': (df[numeric_cols].isnull().sum() / len(df) * 100).round(2)
    })
    print(stats.to_string())
    print()

 BESS - ΣΤΑΤΙΣΤΙΚΑ
                                  Min      Max    Mean     Std  Missing %
Reactive_Power                -84.453   121.55   -0.97   26.09      16.99
Active_Power                 -183.447   196.79   -2.00   24.88      11.55
Apparent_Power                  0.000   213.07   24.70   26.79      16.98
State_of_Charge                 0.000    96.52   28.66   28.92      17.21
Wind_Speed                        NaN      NaN     NaN     NaN     100.00
Weather_Temperature_Celsius    -3.351    45.13   21.93    9.76      23.07
Weather_Relative_Humidity       0.000   102.91   41.65   25.61      23.07
Global_Horizontal_Radiation     0.000  1503.90  265.97  366.84      23.06
Diffuse_Horizontal_Radiation    0.000   757.74   57.37   95.20      23.06
Wind_Direction                -12.242   181.02   32.69   10.74      23.06
Weather_Daily_Rainfall          0.000    27.80    0.33    1.67      23.06
Radiation_Global_Tilted         0.624  1374.17  278.61  372.63      21.57
Radiation_Diffuse_T